In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import os
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from transformers import AutoTokenizer

In [54]:
# Load forum data
with open("../data/rag_text/raw_threads.json", "r", encoding="utf-8") as f:
    forum_data = json.load(f)

# Load YouTube transcript data
with open("../data/rag_text/cleaned_yt_transcripts.json", "r", encoding="utf-8") as f:
    yt_data = json.load(f)

In [55]:
# Helper function to build topic text from forum thread
def build_topic_text(thread):
    parts = [f"Topic: {thread['title']}"]

    for p in thread["posts"]:
        parts.append(
            f"[{p['username']} | {p['created_at']}]"
            f"{p['content_text']}"
        )

    return "\n".join(parts)

In [56]:
# Create Document objects for forum threads
docs = []

for thread in forum_data:
    topic_text = build_topic_text(thread)

    docs.append(
        Document(
            page_content=topic_text,
            metadata={
                "source": "forza_forum",
                "topic_id": thread["topic_id"],
                "topic_slug": thread["topic_slug"],
                "title": thread["title"]
            }
        )
    )

In [57]:
# Create Document objects for YouTube transcripts
yt_docs = []

for v in yt_data:
    yt_docs.append(
        Document(
            page_content=v["text"],
            metadata={
                "source": "youtube",
                "video_id": v["video_id"],
                "domain": "fh5_driving_guides"
            }
        )
    )

In [58]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

In [59]:
all_docs = docs + yt_docs

In [60]:
%%capture
# Text Embedding & Chunking 
embedding_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME, use_fast=False)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    embedding_tokenizer,
    chunk_size=500,
    chunk_overlap=150
)
final_docs = text_splitter.split_documents(all_docs)

In [61]:
# Build embeddings and vector store
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME,
                                   encode_kwargs={"normalize_embeddings": True})

vectorstore = FAISS.from_documents(final_docs, embedding=embeddings)

In [62]:
# Save final chunked documents
with open("../data/rag_text/final_docs.json", "w", encoding="utf-8") as f:
    json.dump(
        [{"page_content": doc.page_content, "metadata": doc.metadata} for doc in final_docs],
        f, ensure_ascii=False, indent=4)

# Create vector database
vectordb_path = "../data/rag_db"
vectorstore.save_local(vectordb_path)

Example Use-Case with k=5 top chunks

In [63]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
question = "Why am I so slow in corners?"
retrieved_docs = retriever.invoke(question)
retrieved_docs

[Document(id='5c01df05-a1f8-4a76-b797-e3c3a6929715', metadata={'source': 'youtube', 'video_id': 'RIEgZ3xlliI', 'domain': 'fh5_driving_guides'}, page_content="online or against your friends. So, I've lined up two clips for you - on the left, it's as you would drive around when you're just playing around trying to be fast but not really trying, and then the one on the right is actual driving lines around the corner.Now you can see that the one on the right is 10 miles per hour faster than the one on the left straight away. You are much faster - 10 miles around the corner. If you add that up around every corner in the whole race, you're going to be miles ahead of any of your friends who are driving the other style.So, this is how you're going to win a race. Let's take a further look into it.What you want to do as you come into the corner is be as far to the apex on the right-hand side as possible to give you the best entry into the corner and exit out of the corner.The way we're going to 